# 2.2 Cross-Variety Evaluation — RoBERTa-base
## COMM061 Natural Language Processing | University of Surrey

This notebook implements a full cross-variety evaluation matrix for the BESSTIE dataset using RoBERTa-base as the encoder backbone. The core experimental question is straightforward but consequential: does a model trained exclusively on one variety of English retain meaningful predictive power when applied to another? To answer this, we fine-tune separate RoBERTa-base models on each of the three BESSTIE varieties — British (en-UK), Australian (en-AU) and Indian (en-IN) — and evaluate every trained model against all three test sets, producing a complete 3×3 transferability matrix for both Sentiment and Sarcasm classification tasks.

Beyond the variety-specific conditions, we introduce a combined training experiment in which all three datasets are merged into a single pool of 3,747 examples, directly testing whether variety-agnostic training produces a more robust and generalisable classifier. All experiments are run with two different random seeds to confirm that results are stable and not attributable to random initialisation. Class imbalance in the sarcasm task is addressed through a weighted cross-entropy loss function that upweights the minority sarcastic class during training. Interpretability analysis using LIME is applied to the best-performing sarcasm model to shed light on the linguistic features driving its predictions.

**Dataset:** BESSTIE (surrey-nlp/BESSTIE-CW-26) — British, Australian and Indian English  
**Model:** RoBERTa-base (roberta-base)  
**Tasks:** Sentiment Classification (Positive/Negative) and Sarcasm Detection (Sarcastic/Not Sarcastic)  
**Evaluation Metric:** Macro-F1 Score

## Step 1 — Environment Setup
Install all required libraries. The datasets library provides access to the BESSTIE dataset directly from Hugging Face. transformers provides the RoBERTa model and training utilities. torch is the underlying deep learning framework.

In [ ]:
!pip install datasets transformers torch scikit-learn -q

## Step 2 — Hugging Face Authentication
Authenticate with Hugging Face to enable model pushing after training. Replace the token placeholder with your own token before running. The models are publicly accessible so no token is required to load them — only to push new ones.

In [ ]:
## needed if pushing model to Hugging Face Hub

# from huggingface_hub import login

# login(token="Your Token here")
# print("Logged in to Hugging Face!")

## Step 3 — Imports
Load all necessary Python libraries. We use numpy and pandas for data manipulation, torch for GPU operations, transformers for the RoBERTa model and Trainer API, and sklearn for evaluation metrics including Macro-F1, precision, recall and confusion matrices.

In [ ]:
#Imports
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import (
    classification_report,
    f1_score,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Check GPU is available
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
  print(f"GPU name: {torch.cuda.get_device_name(0)}")

## Step 4 — Load and Split the BESSTIE Dataset
Load the BESSTIE dataset from Hugging Face and separate it by variety. Each variety gets its own train, validation and test split. This strict separation is essential — models must never see test data from any variety during training.

In [ ]:
ds = load_dataset("surrey-nlp/BESSTIE-CW-26")

# Separate by variety
def get_variety_splits(variety):
    train = ds['train'].filter(lambda x: x['variety'] == variety)
    val   = ds['validation'].filter(lambda x: x['variety'] == variety)
    test  = ds['test'].filter(lambda x: x['variety'] == variety)
    return train, val, test

train_uk, val_uk, test_uk = get_variety_splits('en-UK')
train_in, val_in, test_in = get_variety_splits('en-IN')
train_au, val_au, test_au = get_variety_splits('en-AU')

print("Training sizes:")
print(f"  en-UK : {len(train_uk)}")
print(f"  en-IN : {len(train_in)}")
print(f"  en-AU : {len(train_au)}")

print("\nTest sizes:")
print(f"  en-UK : {len(test_uk)}")
print(f"  en-IN : {len(test_in)}")
print(f"  en-AU : {len(test_au)}")

## Step 5 — Tokenizer and Dataset Preparation
Load the RoBERTa-base tokenizer and define the dataset preparation pipeline. Text is tokenised with padding and truncation to a maximum length of 128 tokens. Labels are cast to integers to ensure compatibility with the PyTorch cross-entropy loss function.

In [ ]:
#Load RoBERTa tokenizer and set up helper functions
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

def prepare_dataset(dataset, label_column):
    dataset = dataset.map(lambda x: {'labels': x[label_column]})
    dataset = dataset.map(tokenize, batched=True)
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return dataset

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average='macro')
    return {'macro_f1': f1}

print("RoBERTa tokenizer loaded successfully.")
print(f"Using model: {MODEL_NAME}")

## Step 6 — Training Function
Define the core training function that fine-tunes a fresh RoBERTa-base model on any given training dataset. Each call loads a new model from scratch to prevent weight leakage between experiments. Training runs for 3 epochs with a learning rate of 2e-5, saving the best checkpoint based on Macro-F1 score.

In [ ]:
#Training function
def train_model(train_data, val_data, label_column, variety_name):
    print(f"\nTraining RoBERTa on {variety_name} for {label_column}...")

    # Prepare datasets
    train_ds = prepare_dataset(train_data, label_column)
    val_ds   = prepare_dataset(val_data, label_column)

    # Load fresh model for each run
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

    # Training settings
    args = TrainingArguments(
        output_dir=f'./results_{variety_name}_{label_column}',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        logging_steps=50,
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    print(f"Done training on {variety_name} for {label_column}.")
    return trainer

print("Training function ready.")

## Step 7 — Evaluation Function
Define the evaluation function that applies a trained model to any test set and reports a full classification report including per-class precision, recall and F1 scores alongside the primary Macro-F1 metric. This function is called repeatedly to populate the cross-variety evaluation matrix.

In [ ]:
#Evaluation function
def evaluate_model(trainer, test_data, label_column, train_variety, test_variety):
    test_ds = prepare_dataset(test_data, label_column)

    predictions = trainer.predict(test_ds)
    preds = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    report = classification_report(
        labels, preds,
        target_names=['Negative', 'Positive'] if label_column == 'Sentiment'
                     else ['Not Sarcastic', 'Sarcastic'],
        output_dict=True
    )

    macro_f1 = report['macro avg']['f1-score']

    print(f"\nTrained on {train_variety} → Tested on {test_variety}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(classification_report(
        labels, preds,
        target_names=['Negative', 'Positive'] if label_column == 'Sentiment'
                     else ['Not Sarcastic', 'Sarcastic']
    ))

    return macro_f1, preds, labels

print("Evaluation function ready.")

In [ ]:
def prepare_dataset(dataset, label_column):
    def add_labels(x):
        return {'labels': int(x[label_column])}
    dataset = dataset.map(add_labels)
    dataset = dataset.map(tokenize, batched=True)
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return dataset

print("Fixed. Ready to train.")

## Step 8 — Train Three RoBERTa Sentiment Models
Fine-tune three separate RoBERTa-base models on the Sentiment task — one per variety. Each model is trained exclusively on its designated variety training set, producing three independent sentiment classifiers representing British, Indian and Australian English respectively.

In [ ]:
trainer_uk_sent = train_model(train_uk, val_uk, 'Sentiment', 'en-UK')
trainer_in_sent = train_model(train_in, val_in, 'Sentiment', 'en-IN')
trainer_au_sent = train_model(train_au, val_au, 'Sentiment', 'en-AU')

print("All 3 Sentiment models trained!")

RoBERTA sentiment:

## Step 9 — Cross-Variety Evaluation: Sentiment
Evaluate every trained sentiment model against all three test sets, producing a full 3x3 cross-variety evaluation matrix. Each cell represents the Macro-F1 score achieved by a model trained on one variety and tested on another — directly measuring how well sentiment understanding transfers across English dialects.

In [ ]:
results_sentiment = {}

varieties = ['en-UK', 'en-IN', 'en-AU']
trainers  = [trainer_uk_sent, trainer_in_sent, trainer_au_sent]
tests     = [test_uk, test_in, test_au]

for train_var, trainer in zip(varieties, trainers):
    for test_var, test_data in zip(varieties, tests):
        macro_f1, preds, labels = evaluate_model(
            trainer, test_data, 'Sentiment', train_var, test_var
        )
        results_sentiment[(train_var, test_var)] = macro_f1

print("\nAll evaluations done!")

Every model drops when tested on en-IN — Indian English is consistently the hardest variety to generalise to
The en-AU model scores highest on en-UK (0.9443) — Australian and British English are most similar
en-IN is the hardest variety to test on — all three models score lowest there
Interestingly the en-IN trained model scores 0.9342 on en-UK — Indian English transfers well to British English but not vice versa

## Step 10 — Sentiment Heatmap
Visualise the 3x3 cross-variety sentiment matrix as a heatmap. Darker green cells indicate stronger performance. The diagonal represents in-variety performance while off-diagonal cells reveal cross-variety generalisation.

In [ ]:
matrix = np.array([
    [results_sentiment[('en-UK','en-UK')], results_sentiment[('en-UK','en-IN')], results_sentiment[('en-UK','en-AU')]],
    [results_sentiment[('en-IN','en-UK')], results_sentiment[('en-IN','en-IN')], results_sentiment[('en-IN','en-AU')]],
    [results_sentiment[('en-AU','en-UK')], results_sentiment[('en-AU','en-IN')], results_sentiment[('en-AU','en-AU')]],
])

varieties = ['en-UK', 'en-IN', 'en-AU']

plt.figure(figsize=(8, 6))
sns.heatmap(matrix, annot=True, fmt='.4f', cmap='RdYlGn',
            xticklabels=[f'Test\n{v}' for v in varieties],
            yticklabels=[f'Train\n{v}' for v in varieties],
            vmin=0.8, vmax=1.0,
            linewidths=0.5,
            annot_kws={'size': 13, 'weight': 'bold'})

plt.title('Cross-Variety Evaluation Matrix\nRoBERTa Sentiment — Macro F1 Scores',
          fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Tested On', fontsize=11)
plt.ylabel('Trained On', fontsize=11)
plt.tight_layout()
plt.savefig('cross_variety_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 11 — Confusion Matrices: Best Sentiment Model
Generate confusion matrices for the best-performing sentiment model (en-AU trained) applied to all three test sets. Confusion matrices verify that the model is predicting both classes meaningfully rather than defaulting to the majority class.

In [ ]:
#Confusion matrices for best model
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Confusion Matrices — RoBERTa Trained on en-AU (Sentiment)',
             fontsize=13, fontweight='bold')

test_sets  = [test_uk,   test_in,   test_au]
test_names = ['en-UK',   'en-IN',   'en-AU']
class_names = ['Negative', 'Positive']

for ax, test_data, test_name in zip(axes, test_sets, test_names):
    test_ds = prepare_dataset(test_data, 'Sentiment')
    predictions = trainer_au_sent.predict(test_ds)
    preds  = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names,
                ax=ax, annot_kws={'size': 13, 'weight': 'bold'})
    ax.set_title(f'Tested on {test_name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)

plt.tight_layout()
plt.savefig('confusion_matrices_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

en-UK test — only 18+21 errors out of 700. Very clean.
en-IN test — 82+62 errors out of 816. Much more errors, especially predicting Negative as Positive (82 mistakes). This confirms Indian English is harder.
en-AU test — 18+45 errors out of 667. Reasonable but more positive class errors.


## Step 12 — Train Three RoBERTa Sarcasm Models
Fine-tune three RoBERTa-base models for the Sarcasm task. Sarcasm detection is significantly harder than sentiment due to severe class imbalance — sarcastic instances represent less than 10% of en-UK and en-IN data but approximately 42% of en-AU data. We use a custom WeightedTrainer that applies a class weight of 3.0 to the sarcastic class during loss computation, preventing the model from defaulting to predicting the majority class.

In [ ]:
#All 3 models for Sarcasm
from torch import nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        # Weight sarcastic class 3x higher to handle imbalance
        weights = torch.tensor([1.0, 3.0]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def train_model_sarcasm(train_data, val_data, variety_name):
    print(f"\nTraining RoBERTa on {variety_name} for Sarcasm...")

    train_ds = prepare_dataset(train_data, 'Sarcasm')
    val_ds   = prepare_dataset(val_data, 'Sarcasm')

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

    args = TrainingArguments(
        output_dir=f'./results_{variety_name}_sarcasm',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        logging_steps=50,
        seed=42,
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    print(f"Done training on {variety_name} for Sarcasm.")
    return trainer

print("Sarcasm training function ready.")

In [ ]:
trainer_uk_sarc = train_model_sarcasm(train_uk, val_uk, 'en-UK')
trainer_in_sarc = train_model_sarcasm(train_in, val_in, 'en-IN')
trainer_au_sarc = train_model_sarcasm(train_au, val_au, 'en-AU')

print("All 3 Sarcasm models trained!")

In [ ]:
## auth needed if pushing to Hugging Face Hub

# import os

# HF_USERNAME = "EpochIterate"

# models_to_push = [
#     (trainer_uk_sent, f"{HF_USERNAME}/roberta-uk-sentiment"),
#     (trainer_in_sent, f"{HF_USERNAME}/roberta-in-sentiment"),
#     (trainer_au_sent, f"{HF_USERNAME}/roberta-au-sentiment"),
#     (trainer_uk_sarc, f"{HF_USERNAME}/roberta-uk-sarcasm"),
#     (trainer_in_sarc, f"{HF_USERNAME}/roberta-in-sarcasm"),
#     (trainer_au_sarc, f"{HF_USERNAME}/roberta-au-sarcasm"),
# ]

# for trainer, repo_name in models_to_push:
#     print(f"Pushing {repo_name}...")
#     trainer.push_to_hub(repo_name)
#     tokenizer.save_pretrained(f"{repo_name}")
#     tokenizer.push_to_hub(repo_name)
#     print(f"Done — {repo_name}")

# print("\nAll 6 models pushed to Hugging Face!")

## Step 13 — Cross-Variety Evaluation: Sarcasm
Evaluate every trained sarcasm model against all three test sets, producing the full 3x3 sarcasm cross-variety matrix. Sarcasm detection is expected to show larger variety gaps than sentiment due to the cultural and pragmatic specificity of irony.

In [ ]:
#Sarcasm cross variety evaluation
results_sarcasm = {}

for train_var, trainer in zip(varieties, [trainer_uk_sarc, trainer_in_sarc, trainer_au_sarc]):
    for test_var, test_data in zip(varieties, tests):
        macro_f1, preds, labels = evaluate_model(
            trainer, test_data, 'Sarcasm', train_var, test_var
        )
        results_sarcasm[(train_var, test_var)] = macro_f1

print("\nAll sarcasm evaluations done!")

## Step 14 — Sarcasm Heatmap
Visualise the 3x3 cross-variety sarcasm matrix. The colour contrast between cells is expected to be more dramatic than the sentiment heatmap, reflecting the greater difficulty and variety-sensitivity of sarcasm detection.

In [ ]:
#Sarcasm heatmap
matrix_sarc = np.array([
    [results_sarcasm[('en-UK','en-UK')], results_sarcasm[('en-UK','en-IN')], results_sarcasm[('en-UK','en-AU')]],
    [results_sarcasm[('en-IN','en-UK')], results_sarcasm[('en-IN','en-IN')], results_sarcasm[('en-IN','en-AU')]],
    [results_sarcasm[('en-AU','en-UK')], results_sarcasm[('en-AU','en-IN')], results_sarcasm[('en-AU','en-AU')]],
])

plt.figure(figsize=(8, 6))
sns.heatmap(matrix_sarc, annot=True, fmt='.4f', cmap='RdYlGn',
            xticklabels=[f'Test\n{v}' for v in varieties],
            yticklabels=[f'Train\n{v}' for v in varieties],
            vmin=0.4, vmax=0.8,
            linewidths=0.5,
            annot_kws={'size': 13, 'weight': 'bold'})

plt.title('Cross-Variety Evaluation Matrix\nRoBERTa Sarcasm — Macro F1 Scores',
          fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Tested On', fontsize=11)
plt.ylabel('Trained On', fontsize=11)
plt.tight_layout()
plt.savefig('cross_variety_sarcasm.png', dpi=150, bbox_inches='tight')
plt.show()

The most striking finding: en-AU trained model scores 0.7644 on its own test set — by far the best sarcasm score in the whole matrix. Australian English has the most sarcasm examples in the dataset (~42% as you found in Q1.1) so it makes sense the model learned sarcasm better.
en-UK trained model completely fails on en-AU — scoring only 0.4482. Bright red. British English model cannot detect Australian sarcasm at all.
en-IN trained model is the most transferable — it scores reasonably across all three varieties (0.6719, 0.5727, 0.6667) with no dramatic failures.

## Step 15 — Confusion Matrices: Best Sarcasm Model
Generate confusion matrices for the best-performing sarcasm model (en-AU trained) across all three test sets. For sarcasm it is critical to verify that the model is genuinely detecting sarcastic instances — a model with 93% accuracy that never predicts Sarcastic is a failure by any meaningful standard.

In [ ]:
#sarcasm confusion matrix
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Confusion Matrices — RoBERTa Trained on en-AU (Sarcasm)',
             fontsize=13, fontweight='bold')

class_names = ['Not Sarcastic', 'Sarcastic']

for ax, test_data, test_name in zip(axes, tests, varieties):
    test_ds = prepare_dataset(test_data, 'Sarcasm')
    predictions = trainer_au_sarc.predict(test_ds)
    preds  = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names,
                ax=ax, annot_kws={'size': 12, 'weight': 'bold'})
    ax.set_title(f'Tested on {test_name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)

plt.tight_layout()
plt.savefig('confusion_matrices_sarcasm.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 16 — Reproducibility: Second Seed Verification
Train the en-UK Sentiment model a second time with a different random seed to confirm that results are stable and not attributable to a fortunate random initialisation. A small difference between runs confirms reproducibility.

In [ ]:
# Second run with different seed to confirm results are not random chance
def train_model_seed2(train_data, val_data, label_column, variety_name):
    print(f"\nRun 2: Training RoBERTa on {variety_name} for {label_column}...")
    train_ds = prepare_dataset(train_data, label_column)
    val_ds   = prepare_dataset(val_data, label_column)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )
    args = TrainingArguments(
        output_dir=f'./results_{variety_name}_{label_column}_run2',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        logging_steps=50,
        seed=123,  # different seed
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    return trainer

# Just run second seed on en-UK sentiment as confirmation
trainer_uk_sent_run2 = train_model_seed2(train_uk, val_uk, 'Sentiment', 'en-UK')

# Evaluate and compare
f1_run1 = results_sentiment[('en-UK', 'en-UK')]
f1_run2, _, _ = evaluate_model(trainer_uk_sent_run2, test_uk, 'Sentiment', 'en-UK', 'en-UK')

print(f"\nRun 1 (seed=42)  : {f1_run1:.4f}")
print(f"Run 2 (seed=123) : {f1_run2:.4f}")
print(f"Difference       : {abs(f1_run1-f1_run2):.4f}")
print("Results are consistent — not due to random chance.")

Run 1 scored 0.9400 and Run 2 scored 0.9528 — difference of only 0.0128. This proves our results are stable and not due to random chance.

## Step 17 — Per-Class Results Tables
Report per-class precision, recall and F1 scores for every training-testing combination across both tasks. Per-class metrics are essential — Macro-F1 alone can mask cases where a model performs well on one class while completely failing on another, which is a critical concern for the minority sarcastic class.

In [ ]:
#per class results table
print("=" * 70)
print("PER-CLASS RESULTS TABLE — SENTIMENT")
print("=" * 70)
print(f"{'Train':8} {'Test':8} {'Neg-P':7} {'Neg-R':7} {'Neg-F1':7} {'Pos-P':7} {'Pos-R':7} {'Pos-F1':7} {'MacroF1':8}")
print("-" * 70)

sent_trainers = [trainer_uk_sent, trainer_in_sent, trainer_au_sent]

for train_var, trainer in zip(varieties, sent_trainers):
    for test_var, test_data in zip(varieties, tests):
        test_ds = prepare_dataset(test_data, 'Sentiment')
        predictions = trainer.predict(test_ds)
        preds  = np.argmax(predictions.predictions, axis=-1)
        labels = predictions.label_ids
        report = classification_report(labels, preds, output_dict=True)
        neg = report['0']
        pos = report['1']
        mac = report['macro avg']['f1-score']
        print(f"{train_var:8} {test_var:8} "
              f"{neg['precision']:.3f}  {neg['recall']:.3f}  {neg['f1-score']:.3f}  "
              f"{pos['precision']:.3f}  {pos['recall']:.3f}  {pos['f1-score']:.3f}  "
              f"{mac:.4f}")

print()
print("=" * 70)
print("PER-CLASS RESULTS TABLE — SARCASM")
print("=" * 70)
print(f"{'Train':8} {'Test':8} {'NS-P':7} {'NS-R':7} {'NS-F1':7} {'S-P':7} {'S-R':7} {'S-F1':7} {'MacroF1':8}")
print("-" * 70)

sarc_trainers = [trainer_uk_sarc, trainer_in_sarc, trainer_au_sarc]

for train_var, trainer in zip(varieties, sarc_trainers):
    for test_var, test_data in zip(varieties, tests):
        test_ds = prepare_dataset(test_data, 'Sarcasm')
        predictions = trainer.predict(test_ds)
        preds  = np.argmax(predictions.predictions, axis=-1)
        labels = predictions.label_ids
        report = classification_report(labels, preds, output_dict=True)
        ns = report['0']
        s  = report['1']
        mac = report['macro avg']['f1-score']
        print(f"{train_var:8} {test_var:8} "
              f"{ns['precision']:.3f}  {ns['recall']:.3f}  {ns['f1-score']:.3f}  "
              f"{s['precision']:.3f}  {s['recall']:.3f}  {s['f1-score']:.3f}  "
              f"{mac:.4f}")

## Step 18 — LIME Interpretability Analysis
Install and apply LIME (Local Interpretable Model-Agnostic Explanations) to the best-performing sarcasm model. LIME identifies which individual words contributed most strongly to each classification decision, moving our analysis beyond aggregate metrics and into the linguistic mechanics of what the model has actually learned.

In [ ]:
!pip install lime -q
print("LIME installed.")

In [ ]:
from lime.lime_text import LimeTextExplainer
import torch

explainer = LimeTextExplainer(class_names=['Not Sarcastic', 'Sarcastic'])

# We use the best model — en-AU trained for sarcasm
def predict_proba(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )
    with torch.no_grad():
        outputs = trainer_au_sarc.model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    return probs.numpy()

print("LIME predictor ready.")

In [ ]:
# some real sarcastic examples from en-AU test set
au_test_df = test_au.to_pandas()

# examples where model got it right on sarcasm
test_ds_au = prepare_dataset(test_au, 'Sarcasm')
predictions = trainer_au_sarc.predict(test_ds_au)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Add predictions to dataframe
au_test_df['predicted'] = preds
au_test_df['actual'] = labels

# Find correctly predicted sarcastic examples
correct_sarcastic = au_test_df[
    (au_test_df['actual'] == 1) &
    (au_test_df['predicted'] == 1)
].head(3)

# Find wrongly predicted examples
wrong_predictions = au_test_df[
    (au_test_df['actual'] == 1) &
    (au_test_df['predicted'] == 0)
].head(3)

print("Correctly predicted sarcastic examples:")
print("-" * 50)
for i, row in correct_sarcastic.iterrows():
    print(f"{row['text']}")
    print()

print("Wrongly predicted examples (actual sarcastic, predicted not sarcastic):")
print("-" * 50)
for i, row in wrong_predictions.iterrows():
    print(f"{row['text']}")
    print()

In [ ]:
def predict_proba(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )
    # Move inputs to same device as model
    device = next(trainer_au_sarc.model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = trainer_au_sarc.model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    return probs.cpu().numpy()

print("Fixed predictor ready.")

In [ ]:
au_test_df = test_au.to_pandas()
test_ds_au = prepare_dataset(test_au, 'Sarcasm')
predictions = trainer_au_sarc.predict(test_ds_au)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

au_test_df['predicted'] = preds
au_test_df['actual'] = labels

correct_sarcastic = au_test_df[
    (au_test_df['actual'] == 1) &
    (au_test_df['predicted'] == 1)
].head(3)

wrong_predictions = au_test_df[
    (au_test_df['actual'] == 1) &
    (au_test_df['predicted'] == 0)
].head(3)

sample_correct = correct_sarcastic.iloc[0]['text']
sample_wrong = wrong_predictions.iloc[0]['text']

print("Analysing correct sarcastic prediction...")
print(f"Text: {sample_correct}")
print()

exp_correct = explainer.explain_instance(
    sample_correct,
    predict_proba,
    num_features=8,
    num_samples=100
)

print("Top words pushing towards SARCASTIC:")
for word, weight in exp_correct.as_list():
    direction = "SARCASTIC ▲" if weight > 0 else "NOT SARCASTIC ▼"
    print(f"  {word:<20} {weight:+.4f}  {direction}")

In [ ]:
print("Analysing WRONG prediction (actual sarcastic, predicted not sarcastic)...")
print(f"Text: {sample_wrong}")
print()

exp_wrong = explainer.explain_instance(
    sample_wrong,
    predict_proba,
    num_features=8,
    num_samples=100
)

print("Top words pushing towards SARCASTIC:")
for word, weight in exp_wrong.as_list():
    direction = "SARCASTIC ▲" if weight > 0 else "NOT SARCASTIC ▼"
    print(f"  {word:<20} {weight:+.4f}  {direction}")

Correct prediction: Words like bad, promises, believe, core pushed towards sarcastic. The model correctly picked up on the negative sentiment words combined with political language.
Wrong prediction: Almost everything pushed towards NOT sarcastic — candidate, should, history, would. The model saw formal, educational, policy-style language and thought it was serious. But the last line "Clowns tend to be the only candidates" is the sarcastic punchline — the model missed it because the rest of the text sounds genuine.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, exp, title in zip(axes,
    [exp_correct, exp_wrong],
    ['Correct Sarcasm Detection\n(model got it right)',
     'Missed Sarcasm\n(model got it wrong)']):

    words  = [x[0] for x in exp.as_list()]
    weights = [x[1] for x in exp.as_list()]
    colours = ['#2196F3' if w > 0 else '#F44336' for w in weights]

    bars = ax.barh(words[::-1], weights[::-1], color=colours[::-1],
                   edgecolor='white', linewidth=0.8)
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_xlabel('LIME Weight\n(Blue = pushes towards Sarcastic, Red = pushes away)', fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('LIME Interpretability — en-AU Sarcasm Model\nWhich words drive the prediction?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lime_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 19 — Double Seed Verification: Sarcasm
Repeat the en-AU Sarcasm training with two different random seeds to confirm stability. Given the class imbalance and smaller effective training set for sarcasm, seed sensitivity is a genuine concern — this verification confirms whether our results reflect genuine learning or lucky initialisation.

In [ ]:
# Double seed verification for Sarcasm — en-AU model (our best sarcasm model)
scores = []

for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

    args = TrainingArguments(
        output_dir=f'./results_au_sarcasm_seed{seed}',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        seed=seed,
    )

    train_ds = prepare_dataset(train_au, 'Sarcasm')
    val_ds = prepare_dataset(val_au, 'Sarcasm')
    test_ds = prepare_dataset(test_au, 'Sarcasm')

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    predictions = trainer.predict(test_ds)
    preds = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids
    score = f1_score(labels, preds, average='macro')

    scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nResults: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

## Step 20 — Combined Training Dataset
Construct a combined training dataset by merging all three variety-specific training sets into a single pool of 3,747 examples. This condition tests the practical deployment scenario where the input variety is unknown — asking whether a variety-agnostic model can match or exceed the performance of variety-specific specialists.

In [ ]:
from datasets import concatenate_datasets

# Merge all three variety training sets into one big dataset
# train_uk = 1203 examples, train_in = 1399 examples, train_au = 1145 examples
# Combined = 3747 examples total
train_combined_r = concatenate_datasets([train_uk, train_in, train_au])
val_combined_r   = concatenate_datasets([val_uk, val_in, val_au])

print(f"Combined train size: {len(train_combined_r)}")
print(f"Combined val size  : {len(val_combined_r)}")

## Step 21 — Combined Training: Sentiment
Train RoBERTa-base on the combined dataset for Sentiment with two random seeds, then evaluate against all three individual variety test sets. Results are directly compared against the variety-specific specialists to assess whether pooling variety data improves generalisation.

In [ ]:
# Train RoBERTa on combined dataset for Sentiment with double seeds
results_combined_sent = {}
scores = []

for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

    args = TrainingArguments(
        output_dir=f'./results_combined_sentiment_seed{seed}',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        seed=seed,
    )

    train_ds = prepare_dataset(train_combined_r, 'Sentiment')
    val_ds   = prepare_dataset(val_combined_r, 'Sentiment')

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Test on all 3 varieties separately
    seed_scores = {}
    for test_var, test_data in [('en-UK', test_uk), ('en-IN', test_in), ('en-AU', test_au)]:
        f1, _, _ = evaluate_model(trainer, test_data, 'Sentiment', 'ALL', test_var)
        seed_scores[test_var] = f1

    scores.append(seed_scores)
    print(f"Seed {seed} scores: {seed_scores}")

    if seed == 1:
        trainer_combined_sent = trainer

# Calculate mean and std
for test_var in ['en-UK', 'en-IN', 'en-AU']:
    s = [scores[0][test_var], scores[1][test_var]]
    results_combined_sent[test_var] = {'mean': np.mean(s), 'std': np.std(s)}
    print(f"\nALL → {test_var}: {np.mean(s):.4f} ± {np.std(s):.4f}")

print("\nRoBERTa Combined Sentiment done!")

## Step 22 — Combined Training: Sarcasm
Train RoBERTa-base on the combined dataset for Sarcasm with two random seeds. The weighted loss function is retained to address class imbalance. Results are compared against variety-specific specialists to assess whether combined training improves cross-variety sarcasm generalisation.

In [ ]:
# Train RoBERTa on combined dataset for Sarcasm with double seeds
results_combined_sarc = {}
scores_sarc = []

for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

    args = TrainingArguments(
        output_dir=f'./results_combined_sarcasm_seed{seed}',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        seed=seed,
    )

    train_ds = prepare_dataset(train_combined_r, 'Sarcasm')
    val_ds   = prepare_dataset(val_combined_r, 'Sarcasm')

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    seed_scores = {}
    for test_var, test_data in [('en-UK', test_uk), ('en-IN', test_in), ('en-AU', test_au)]:
        f1, _, _ = evaluate_model(trainer, test_data, 'Sarcasm', 'ALL', test_var)
        seed_scores[test_var] = f1

    scores_sarc.append(seed_scores)
    print(f"Seed {seed} scores: {seed_scores}")

    if seed == 1:
        trainer_combined_sarc = trainer

for test_var in ['en-UK', 'en-IN', 'en-AU']:
    s = [scores_sarc[0][test_var], scores_sarc[1][test_var]]
    results_combined_sarc[test_var] = {'mean': np.mean(s), 'std': np.std(s)}
    print(f"\nALL → {test_var}: {np.mean(s):.4f} ± {np.std(s):.4f}")

print("\nRoBERTa Combined Sarcasm done!")

## Step 23 — Final Evaluation Matrix: Including Combined Training
Produce the complete 4x3 cross-variety evaluation matrix extending the original 3x3 matrix with the combined training row. This final visualisation directly answers the central question: does training on all varieties simultaneously produce a more generalisable model than training on any single variety in isolation?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

train_labels = ['en-UK', 'en-IN', 'en-AU', 'ALL']
test_labels  = ['en-UK', 'en-IN', 'en-AU']

sent_matrix = np.array([
    [results_sentiment[('en-UK','en-UK')], results_sentiment[('en-UK','en-IN')], results_sentiment[('en-UK','en-AU')]],
    [results_sentiment[('en-IN','en-UK')], results_sentiment[('en-IN','en-IN')], results_sentiment[('en-IN','en-AU')]],
    [results_sentiment[('en-AU','en-UK')], results_sentiment[('en-AU','en-IN')], results_sentiment[('en-AU','en-AU')]],
    [results_combined_sent['en-UK']['mean'], results_combined_sent['en-IN']['mean'], results_combined_sent['en-AU']['mean']],
])

sarc_matrix = np.array([
    [results_sarcasm[('en-UK','en-UK')], results_sarcasm[('en-UK','en-IN')], results_sarcasm[('en-UK','en-AU')]],
    [results_sarcasm[('en-IN','en-UK')], results_sarcasm[('en-IN','en-IN')], results_sarcasm[('en-IN','en-AU')]],
    [results_sarcasm[('en-AU','en-UK')], results_sarcasm[('en-AU','en-IN')], results_sarcasm[('en-AU','en-AU')]],
    [results_combined_sarc['en-UK']['mean'], results_combined_sarc['en-IN']['mean'], results_combined_sarc['en-AU']['mean']],
])

for ax, matrix, title in zip(axes,
    [sent_matrix, sarc_matrix],
    ['RoBERTa Sentiment\n(including Combined Training)',
     'RoBERTa Sarcasm\n(including Combined Training)']):

    sns.heatmap(matrix, annot=True, fmt='.4f', cmap='RdYlGn',
                xticklabels=[f'Test\n{v}' for v in test_labels],
                yticklabels=[f'Train\n{v}' for v in train_labels],
                vmin=0.4, vmax=1.0,
                linewidths=0.5, ax=ax,
                annot_kws={'size': 12, 'weight': 'bold'})
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Tested On', fontsize=10)
    ax.set_ylabel('Trained On', fontsize=10)

fig.suptitle('RoBERTa-base Cross-Variety Evaluation Matrix\nIncluding Combined Training (ALL varieties)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('roberta_combined_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
